In [19]:
import dlt

from dlt.destinations import filesystem
from dlt.destinations.adapters import iceberg_adapter, iceberg_partition

from datetime import datetime, timezone
from uuid import uuid4
from mongodb import mongodb

In [20]:
DATABASE_NAME = "test"

SOURCE_SYSTEM = "mongodb_atlas"
INGESTION_TYPE = "incremental"
INITIAL_CURSOR = datetime(1970, 1, 1, tzinfo=timezone.utc)

INGESTED_AT = datetime.now(timezone.utc)
RUN_ID = str(uuid4())

BRONZE_BUCKET_URL = "s3://book-library-lakehouse/bronze"

destination = filesystem(
    bucket_url=BRONZE_BUCKET_URL
)

iceberg_partition.day("_ingested_at")

PartitionSpec(source_column='_ingested_at', transform='day', param_value=None, partition_field=None)

In [21]:
source = mongodb(
    database=DATABASE_NAME
).with_resources("authors", "books")

In [26]:
for resource_name, resource in source.resources.items():

    resource.apply_hints(
        incremental=dlt.sources.incremental(
            "updatedAt",
            initial_value=INITIAL_CURSOR
        ),
        write_disposition="append",
        primary_key="_id"
    )

    def add_metadata(item, resource_name=resource_name):
        item = dict(item)

        item["_source_system"] = SOURCE_SYSTEM
        item["_source_resource"] = resource_name
        item["_ingestion_type"] = INGESTION_TYPE
        item["_ingested_at"] = INGESTED_AT
        item["_load_id"] = RUN_ID

        return item

    resource.add_map(add_metadata)

    # particioniranje po dnevu vnosa
    iceberg_adapter(
        resource,
        partition=[
            iceberg_partition.day("_ingested_at")
        ]
    )

In [27]:
pipeline = dlt.pipeline(
    pipeline_name="mongodb_iceberg",
    destination=destination,
    dataset_name="mongodb_atlas"
)

In [28]:
load_info = pipeline.run(
    source,
    write_disposition="append",
    table_format="iceberg"
)

print("MongoDB - Iceberg BRONZE ... DONE")
print(load_info)
print("RUN ID:", RUN_ID)
print("INGESTED AT:", INGESTED_AT)

2026-11-03 11:53:35,146|[WARNING]|24696|35852|dlt|validate.py|verify_normalized_table:113|In schema `mongodb`: The following columns in table 'authors' did not receive any data during this load and therefore could not have their types inferred:
  - _source_resource

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'_source_resource': {'data_type': 'text'}})

2026-11-03 11:53:35,146|[WARNING]|24696|35852|dlt|validate.py|verify_normalized_table:113|In schema `mongodb`: The following columns in table 'books' did not receive any data during this load and therefore could not have their types inferred:
  - _source_resource

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For exam

MongoDB - Iceberg BRONZE ... DONE
Pipeline mongodb_iceberg load step finished in 1.63 seconds
1 load package(s) were loaded to destination filesystem and into dataset mongodb_atlas
The filesystem destination used s3://book-library-lakehouse/bronze location to store data
Load package 1793703214.8388884 is LOADED and contains no failed jobs
RUN ID: c229fdb8-6314-4c25-8a8c-25b1c43d9d28
INGESTED AT: 2026-11-03 10:26:49.860331+00:00


In [25]:
baseline_snapshots = {}

from pyiceberg.catalog import load_catalog

catalog = load_catalog(    "book_library",
    **{
        "type": "sql",
        "uri": "sqlite:///./catalog/iceberg_catalog.db",
        "warehouse": "s3://book-library-lakehouse/warehouse",

        "s3.endpoint": "http://127.0.0.1:9000",
        "s3.access-key-id": "nelacopot",
        "s3.secret-access-key": "nelacopot",
        "s3.region": "us-east-1",
        "s3.force-virtual-addressing": "false",

        "py-io-impl": "pyiceberg.io.pyarrow.PyArrowFileIO"
    })

for table_name in ["authors", "books"]:

    table = catalog.load_table(
        ("mongodb_atlas", table_name)
    )

    snapshot = table.current_snapshot()

    baseline_snapshots[table_name] = snapshot.snapshot_id

    old_data = table.scan(
        snapshot_id=snapshot.snapshot_id
    ).to_arrow()

    print("\n------------------------------")
    print("Tabela:", table_name)
    print("Začetni snapshot ID:", snapshot.snapshot_id)
    print("Začetno število zapisov:", old_data.num_rows)


------------------------------
Tabela: authors
Začetni snapshot ID: 5544653622342132227
Začetno število zapisov: 2006

------------------------------
Tabela: books
Začetni snapshot ID: 6859439240913658252
Začetno število zapisov: 9908


Po dodanem enem zapisu knjige in enem zapisu avtorja:

In [29]:
for table_name in ["authors", "books"]:

    table = catalog.load_table(
        ("mongodb_atlas", table_name)
    )

    old_snapshot_id = baseline_snapshots[table_name]

    old_data = table.scan(
        snapshot_id=old_snapshot_id
    ).to_arrow()

    new_data = table.scan().to_arrow()

    print("Stari snapshot:", old_snapshot_id)
    print("Staro število zapisov:", old_data.num_rows)

    print(
        "Novi snapshot:",
        table.current_snapshot().snapshot_id
    )
    print("Novo število zapisov:", new_data.num_rows)


    print("\nZgodovina snapshotov:")

    snapshots = table.inspect.snapshots().to_pylist()

    for snapshot in snapshots:
        print(
            "Čas:", snapshot["committed_at"],
            "| ID:", snapshot["snapshot_id"]
        )

Stari snapshot: 5544653622342132227
Staro število zapisov: 2006
Novi snapshot: 6161922051348093618
Novo število zapisov: 2007

Zgodovina snapshotov:
Čas: 2026-11-03 10:26:20.072000 | ID: 5544653622342132227
Čas: 2026-11-03 10:53:36.656000 | ID: 6161922051348093618
Stari snapshot: 6859439240913658252
Staro število zapisov: 9908
Novi snapshot: 4237246761017498356
Novo število zapisov: 9909

Zgodovina snapshotov:
Čas: 2026-11-03 10:26:20.072000 | ID: 6859439240913658252
Čas: 2026-11-03 10:53:36.656000 | ID: 4237246761017498356
